# TC4NPP two-day Polyphemus scenarios

Run the full-year 48-hour screening cases first, then run the four retained events in each meteorological regime. The full-year results were used to select the final 16 event dates.

In [ ]:
# @File: 2days.ipynb
# @Date: 2026-06-28
# @Author: Lubber
# @Description: Run full-year screening and the final four-regime scenarios.

import os
import sys
from datetime import date, timedelta
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.config import BASE_DIR, SCENE_DIR, NPP, SITE_NAME
from src.data_preprocessor import DataPreprocessor
from src.model_runner import ModelRunner

In [ ]:
TOTAL_DAYS = 2
NPP_INFO = {"name": SITE_NAME, **NPP[SITE_NAME]}

# Full-year screening: run once with RUN_FULL_YEAR = True.
RUN_FULL_YEAR = False
FULL_YEAR_PREPROCESS = False
FULL_YEAR_START = date(2023, 1, 1)
FULL_YEAR_END = date(2023, 12, 29)
FULL_YEAR_RELEASE_DURATION = 48

# Final scenarios retained from the full-year screening.
RUN_PAPER_SCENARIOS = True
PAPER_PREPROCESS = True
PAPER_RELEASE_DURATIONS = [12, 24, 36, 48]
SELECTED_EVENTS = {
    "BG": [
        "20230103170000",
        "20230306220000",
        "20230429140000",
        "20230917090000",
    ],
    "SW": [
        "20230219080000",
        "20230321000000",
        "20230528220000",
        "20230704100000",
    ],
    "HR": [
        "20230325040000",
        "20230405100000",
        "20230613180000",
        "20230816050000",
    ],
    "TC": [
        "20230727150000",
        "20230831040000",
        "20230903190000",
        "20231006010000",
    ],
}

In [ ]:
def run_case(start_time: str, release_durations: list[int], preprocess: bool) -> None:
    """Preprocess meteorology and run Polair3D for one event."""
    if preprocess:
        os.chdir(SCENE_DIR)
        preprocessor = DataPreprocessor(start_time, TOTAL_DAYS + 1)
        preprocessor.handle_land_use_data()
        if not preprocessor.handle_meteo_data():
            raise RuntimeError(f"Meteorological preprocessing failed for {start_time}")
        os.chdir(BASE_DIR)

    for release_duration in release_durations:
        os.chdir(SCENE_DIR)
        runner = ModelRunner(start_time, TOTAL_DAYS, NPP_INFO, release_duration)
        runner.run_model()
        os.chdir(BASE_DIR)

In [ ]:
if RUN_FULL_YEAR:
    current_date = FULL_YEAR_START
    while current_date <= FULL_YEAR_END:
        start_time = current_date.strftime("%Y%m%d")
        run_case(
            start_time,
            [FULL_YEAR_RELEASE_DURATION],
            FULL_YEAR_PREPROCESS,
        )
        current_date += timedelta(days=1)

In [ ]:
if RUN_PAPER_SCENARIOS:
    for event_type, event_times in SELECTED_EVENTS.items():
        for start_time in event_times:
            run_case(start_time, PAPER_RELEASE_DURATIONS, PAPER_PREPROCESS)

    event_count = sum(len(times) for times in SELECTED_EVENTS.values())
    print(f"Processed {event_count} events and {len(PAPER_RELEASE_DURATIONS)} release durations.")